# Ablation, importance, cost and the leakage audit (Prompt #9, Steps 29-34, 47-53, 60-64)

Cumulative family ablation (statistics, then + regression, + OU, ...), leave one family out, block permutation importance with correlated features permuted together, the interaction hierarchy check, the live cost of each set, streaming reconstruction, the leakage audit and the reserved-period isolation check. Caveat: the ablation pool is the quality universe without the pipeline veto, so residual-target rows include mechanical effects (invariant 9).

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.selection` and `xauusd_quant.research.feature_selection_*` and read back from `results/feature_selection/`. Generate those first (needs the Prompt #8 research of the same timeframe for the pipeline-null veto):

```
xq feature-select --timeframe 5m                        # every stage + report + manifests
xq feature-select --timeframe 5m --target residual_reduction
xq feature-select --timeframe 5m --target future_return --horizon 5
xq feature-selection-report --timeframe 5m              # summary, SEL-H ledger rows, plots
xq build-feature-manifest --set standard --timeframe 5m # verify + reproduce the matrix
```

**Periods.** Development 2003-2017 selects; validation 2018-2021 only evaluates what development chose and reads the plateaus; the reserved test period (2022-) is never loaded with outcomes. Diagnostic models are linear (ridge, Lasso, elastic net, L1 logistic) on training-CDF rank designs - not the ML system of Prompt #10, and nothing here is a trading rule.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.selection.config import load_selection_config

scfg = load_selection_config(ROOT / "config" / "feature_selection.yaml")

TIMEFRAME = "5m"       # 5m | 15m (configured); 30m | 1h if they were run

out = scfg.results_path / TIMEFRAME
if not out.exists():
    print(f"No selection for {TIMEFRAME}. Run:  xq feature-select --timeframe {TIMEFRAME}")

def table(name):
    """Load a previously generated selection table (None if it was not written)."""
    path = out / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name):
    path = out / "plots" / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"(no plot {path.name})")

def load_json(name):
    path = out / name
    return json.loads(path.read_text()) if path.exists() else {}

summary = load_json("summary.json")
sets = load_json("sets.json")
{k: v for k, v in summary.items() if not isinstance(v, (dict, list))}

## Family ablation

In [ ]:
plot('family_composition')
table('family_ablation').filter(pl.col('split') == 'development_to_validation').pivot(on='target', index='step', values='rank_ic')

## Leave one family out

In [ ]:
pl.DataFrame(summary.get('leave_one_family_out', []))

In [ ]:
table('leave_one_family_out').filter(pl.col('split') == 'development_to_validation').sort('delta_rank_ic')

## Permutation importance (block permutation, development -> validation)

In [ ]:
table('permutation_importance').sort('importance_rank_ic', descending=True).head(25)

## Interactions: increment over their components

In [ ]:
table('interactions_review').filter(pl.col('split') == 'development_to_validation')

## Cost, warm-up and buffer per set

In [ ]:
table('cost')

## Live reconstruction and leakage audit

In [ ]:
load_json('live_reconstruction.json')

In [ ]:
audit = load_json('leakage_audit.json')
print('all verified:', audit.get('all_verified'))
pl.DataFrame(audit.get('features', []))

## Reserved-period isolation

In [ ]:
load_json('reserved_isolation.json')

## Every selection experiment counted (SEL-H ledger)

In [ ]:
summary.get('trials')